# Abundant regularization sweep — final Gini-refinement analyzer

This notebook analyzes the completed coarse sweep for graph regularization in
the abundant-data regime.

It performs **analysis only**:

- no environment rollouts,
- no graph-identifier training,
- no cache reconstruction.

It expects results from:

`experiment_2026_09_25_abundant_regularization_sweep`

and compares:

- **none** — finalized September model;
- **theta_l2** — paper-inspired \(L_2(\Theta)\);
- **entropy** — normalized row-entropy penalty;
- **gini** — normalized row-Gini impurity penalty.

The main tuning objective is not merely "make \(A\) sparse". We inspect jointly:

1. weighted-\(A\) recovery;
2. support/ranking recovery;
3. graph concentration;
4. one-step prediction;
5. downstream control.

A useful regularizer should improve graph recovery without materially degrading
control.

### Theta-L2 refinement

This version expects the refined paper-style grid $10^{-8},10^{-7},10^{-6},10^{-5},10^{-4}$, while retaining the existing entropy and Gini conditions.

### Gini refinement grid

The final Gini grid is $\{10^{-6},10^{-5},3\times10^{-5},5\times10^{-5},10^{-4},2\times10^{-4},3\times10^{-4},5\times10^{-4},10^{-3}\}$.


In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import t as student_t

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 320)
pd.set_option("display.max_rows", 300)

def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "opinion_dynamics").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing opinion_dynamics/.")

REPO_ROOT = find_repo_root()
RESULTS_DIR = (
    REPO_ROOT
    / "opinion_dynamics"
    / "experiments"
    / "results"
    / "experiment_2026_09_25_abundant_regularization_sweep"
)

ALL_RUNS = RESULTS_DIR / "regularization_sweep_all_runs.csv"
COMPLETION = RESULTS_DIR / "COMPUTATION_COMPLETE.json"

for path in [ALL_RUNS, COMPLETION]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Repository:", REPO_ROOT)
print("Results:", RESULTS_DIR)


## Completion and coverage audit

In [ ]:
completion = json.loads(COMPLETION.read_text(encoding="utf-8"))
display(pd.Series(completion, name="value").to_frame())

assert completion["status"] == "success"
assert completion["study_name"] == "abundant_regularization_sweep_unbounded_lambda1"
assert int(completion["n_environments"]) == 195
assert int(completion["learning_seed"]) == 0
assert int(completion["n_regularized_variants"]) == 18
assert int(completion["n_new_fits"]) == 975
assert int(completion["n_reused_fits"]) == 2535

df = pd.read_csv(ALL_RUNS)

required = {
    "environment_id", "scenario_class", "family", "dynamics",
    "regularizer_kind", "regularizer_lambda",
    "mean_end", "mean_over_trajectory",
    "model_over_identity",
    "A_MAE", "A_Fro", "row_TV",
    "true_edge_mass", "support_topdegree_recall", "edge_AP",
    "centrality_L1", "row_entropy_norm", "effective_neighbors",
}
missing_cols = sorted(required - set(df.columns))
if missing_cols:
    raise RuntimeError(f"Missing required columns: {missing_cols}")

df["regularizer_lambda"] = df["regularizer_lambda"].astype(float)

conditions = (
    df[["regularizer_kind", "regularizer_lambda"]]
    .drop_duplicates()
    .sort_values(["regularizer_kind", "regularizer_lambda"])
    .reset_index(drop=True)
)

coverage = (
    df.groupby(["regularizer_kind", "regularizer_lambda"])
    .agg(rows=("environment_id", "size"),
         environments=("environment_id", "nunique"))
    .reset_index()
    .sort_values(["regularizer_kind", "regularizer_lambda"])
)

print("Conditions:", len(conditions))
print("Rows:", len(df))
display(coverage)

assert len(conditions) == 19
assert len(df) == 195 * 19
assert (coverage["rows"] == 195).all()
assert (coverage["environments"] == 195).all()

print("\nCOVERAGE AUDIT PASSED: 19 conditions × 195 paired environments.")


## Overall sweep summary

Main directions:

- `mean_end`: higher is better;
- `model_over_identity`: lower is better;
- `row_TV`: lower is better;
- `true_edge_mass`: higher is better;
- `edge_AP`: higher is better;
- `centrality_L1`: lower is better;
- `row_entropy_norm`: lower means more concentrated rows;
- `effective_neighbors`: lower means more concentrated rows.


In [ ]:
ref = df[
    (df["regularizer_kind"] == "none")
    & (df["regularizer_lambda"] == 0.0)
].set_index("environment_id")
assert len(ref) == 195

metric_cols = [
    "mean_end", "mean_over_trajectory", "model_over_identity",
    "row_TV", "true_edge_mass", "support_topdegree_recall",
    "edge_AP", "centrality_L1", "row_entropy_norm", "effective_neighbors",
]

work = df.copy()
for metric in metric_cols:
    work[f"delta_{metric}_vs_none"] = work.apply(
        lambda r: float(r[metric]) - float(ref.loc[r["environment_id"], metric]),
        axis=1,
    )

summary = (
    work.groupby(["regularizer_kind", "regularizer_lambda"], as_index=False)
    .agg(
        n=("environment_id", "size"),
        mean_end=("mean_end", "mean"),
        mean_over_trajectory=("mean_over_trajectory", "mean"),
        prediction_ratio=("model_over_identity", "mean"),
        row_TV=("row_TV", "mean"),
        true_edge_mass=("true_edge_mass", "mean"),
        support_recall=("support_topdegree_recall", "mean"),
        edge_AP=("edge_AP", "mean"),
        centrality_L1=("centrality_L1", "mean"),
        row_entropy=("row_entropy_norm", "mean"),
        effective_neighbors=("effective_neighbors", "mean"),
        delta_control=("delta_mean_end_vs_none", "mean"),
        delta_prediction_ratio=("delta_model_over_identity_vs_none", "mean"),
        delta_row_TV=("delta_row_TV_vs_none", "mean"),
        delta_true_edge_mass=("delta_true_edge_mass_vs_none", "mean"),
        delta_edge_AP=("delta_edge_AP_vs_none", "mean"),
        delta_centrality_L1=("delta_centrality_L1_vs_none", "mean"),
    )
)

display(summary.sort_values(["regularizer_kind", "regularizer_lambda"]).round(6))


## Best observed setting within each regularizer family

In [ ]:
regularized = summary[summary["regularizer_kind"] != "none"].copy()
best_rows = []

for kind, g in regularized.groupby("regularizer_kind"):
    best_rows.append({
        "regularizer_kind": kind,
        "lambda_lowest_row_TV": float(g.loc[g["row_TV"].idxmin(), "regularizer_lambda"]),
        "lowest_row_TV": float(g["row_TV"].min()),
        "lambda_highest_true_edge_mass": float(g.loc[g["true_edge_mass"].idxmax(), "regularizer_lambda"]),
        "highest_true_edge_mass": float(g["true_edge_mass"].max()),
        "lambda_highest_edge_AP": float(g.loc[g["edge_AP"].idxmax(), "regularizer_lambda"]),
        "highest_edge_AP": float(g["edge_AP"].max()),
        "lambda_highest_control": float(g.loc[g["mean_end"].idxmax(), "regularizer_lambda"]),
        "highest_control": float(g["mean_end"].max()),
        "lambda_lowest_prediction_ratio": float(g.loc[g["prediction_ratio"].idxmin(), "regularizer_lambda"]),
        "lowest_prediction_ratio": float(g["prediction_ratio"].min()),
    })

best_by_family = pd.DataFrame(best_rows)
display(best_by_family.round(6))

baseline = summary[summary["regularizer_kind"] == "none"].iloc[0]
print("\nNo-regularization reference:")
display(
    baseline[
        ["mean_end", "prediction_ratio", "row_TV", "true_edge_mass",
         "edge_AP", "centrality_L1", "row_entropy", "effective_neighbors"]
    ].to_frame("none")
)


## Paired uncertainty versus no regularization

Structured cases use a paired Student-\(t\) interval.

Generic dynamics use the same crossed \(5\times5\) topology/opinion bootstrap
logic as the canonical paper analysis.


In [ ]:
def student_t_ci(values: np.ndarray, confidence: float = 0.95):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    mean = float(x.mean())
    if len(x) <= 1:
        return mean, np.nan, np.nan
    crit = float(student_t.ppf(0.5 + confidence / 2, df=len(x) - 1))
    half = crit * float(x.std(ddof=1)) / math.sqrt(len(x))
    return mean, mean - half, mean + half

def crossed_bootstrap_ci(group: pd.DataFrame, col: str, n_boot: int = 20_000, seed: int = 20260926):
    pivot = group.pivot(index="topology_seed", columns="opinion_seed", values=col)
    if pivot.shape != (5, 5) or pivot.isna().any().any():
        raise ValueError(f"Incomplete 5x5 crossed design for {col}: {pivot.shape}")
    matrix = pivot.to_numpy(float)
    rng = np.random.default_rng(seed)
    r = rng.integers(0, 5, size=(n_boot, 5))
    c = rng.integers(0, 5, size=(n_boot, 5))
    sampled = matrix[r[:, :, None], c[:, None, :]]
    boot = sampled.mean(axis=(1, 2))
    lo, hi = np.quantile(boot, [0.025, 0.975])
    return float(matrix.mean()), float(lo), float(hi)

paired_rows = []

for (kind, lam), candidate in work[
    work["regularizer_kind"] != "none"
].groupby(["regularizer_kind", "regularizer_lambda"], sort=False):
    for (scenario, dynamics), g in candidate.groupby(
        ["scenario_class", "dynamics"], sort=False
    ):
        for metric in ["mean_end", "row_TV", "true_edge_mass", "edge_AP", "centrality_L1"]:
            delta_col = f"delta_{metric}_vs_none"
            if (
                scenario == "unstructured_random"
                and len(g) == 25
                and g["topology_seed"].nunique() == 5
                and g["opinion_seed"].nunique() == 5
            ):
                mean, lo, hi = crossed_bootstrap_ci(g, delta_col)
                method = "crossed bootstrap"
            else:
                mean, lo, hi = student_t_ci(g[delta_col].to_numpy())
                method = "paired Student-t"

            paired_rows.append({
                "regularizer_kind": kind,
                "regularizer_lambda": float(lam),
                "scenario_class": scenario,
                "dynamics": dynamics,
                "metric": metric,
                "delta_vs_none_mean": mean,
                "ci_low": lo,
                "ci_high": hi,
                "ci_method": method,
                "n_environments": len(g),
            })

paired_ci = pd.DataFrame(paired_rows)

print("Control delta vs none:")
display(
    paired_ci[paired_ci["metric"] == "mean_end"]
    .sort_values(["scenario_class", "dynamics", "regularizer_kind", "regularizer_lambda"])
    .round(6)
)


## Structured HK — primary stress test

In [ ]:
hk = (
    work[
        (work["scenario_class"] == "structured_mechanism")
        & (work["dynamics"] == "hegselmannkrause")
    ]
    .groupby(["regularizer_kind", "regularizer_lambda"], as_index=False)
    .agg(
        n=("environment_id", "size"),
        control=("mean_end", "mean"),
        prediction_ratio=("model_over_identity", "mean"),
        row_TV=("row_TV", "mean"),
        true_edge_mass=("true_edge_mass", "mean"),
        support_recall=("support_topdegree_recall", "mean"),
        edge_AP=("edge_AP", "mean"),
        centrality_L1=("centrality_L1", "mean"),
        row_entropy=("row_entropy_norm", "mean"),
        effective_neighbors=("effective_neighbors", "mean"),
        delta_control=("delta_mean_end_vs_none", "mean"),
        delta_row_TV=("delta_row_TV_vs_none", "mean"),
    )
)

display(hk.sort_values(["regularizer_kind", "regularizer_lambda"]).round(6))

print("\nStructured-HK paired control intervals:")
display(
    paired_ci[
        (paired_ci["scenario_class"] == "structured_mechanism")
        & (paired_ci["dynamics"] == "hegselmannkrause")
        & (paired_ci["metric"] == "mean_end")
    ].sort_values(["regularizer_kind", "regularizer_lambda"]).round(6)
)

print("\nStructured-HK paired row-TV intervals:")
display(
    paired_ci[
        (paired_ci["scenario_class"] == "structured_mechanism")
        & (paired_ci["dynamics"] == "hegselmannkrause")
        & (paired_ci["metric"] == "row_TV")
    ].sort_values(["regularizer_kind", "regularizer_lambda"]).round(6)
)


## Full dynamics breakdown

In [ ]:
by_dyn = (
    work.groupby(
        ["regularizer_kind", "regularizer_lambda", "scenario_class", "dynamics"],
        as_index=False,
    )
    .agg(
        n=("environment_id", "size"),
        control=("mean_end", "mean"),
        prediction_ratio=("model_over_identity", "mean"),
        row_TV=("row_TV", "mean"),
        true_edge_mass=("true_edge_mass", "mean"),
        support_recall=("support_topdegree_recall", "mean"),
        edge_AP=("edge_AP", "mean"),
        centrality_L1=("centrality_L1", "mean"),
        effective_neighbors=("effective_neighbors", "mean"),
        delta_control=("delta_mean_end_vs_none", "mean"),
        delta_row_TV=("delta_row_TV_vs_none", "mean"),
        delta_edge_AP=("delta_edge_AP_vs_none", "mean"),
    )
)

for scenario in ["structured_mechanism", "unstructured_random"]:
    print("\n", "=" * 90)
    print(scenario)
    print("=" * 90)
    display(
        by_dyn[by_dyn["scenario_class"] == scenario]
        .sort_values(["dynamics", "regularizer_kind", "regularizer_lambda"])
        .round(6)
    )


## Pareto frontier: weighted-\(A\) recovery vs control

A condition is Pareto-efficient if no other condition has both lower/equal
row-TV and higher/equal control, with at least one strict improvement.


In [ ]:
points = summary.copy()
pareto_idx = []

for i, r in points.iterrows():
    dominated = False
    for j, q in points.iterrows():
        if i == j:
            continue
        if (
            q["row_TV"] <= r["row_TV"]
            and q["mean_end"] >= r["mean_end"]
            and (
                q["row_TV"] < r["row_TV"]
                or q["mean_end"] > r["mean_end"]
            )
        ):
            dominated = True
            break
    if not dominated:
        pareto_idx.append(i)

pareto = points.loc[pareto_idx].sort_values("row_TV")

display(
    pareto[
        ["regularizer_kind", "regularizer_lambda", "row_TV",
         "true_edge_mass", "edge_AP", "mean_end",
         "prediction_ratio", "effective_neighbors"]
    ].round(6)
)


## Sweep curves

In [ ]:
family_order = ["theta_l2", "entropy", "gini"]

for metric, ylabel, lower_better in [
    ("row_TV", "Mean row-TV error of A", True),
    ("true_edge_mass", "Mass assigned to true edges", False),
    ("edge_AP", "True-edge average precision", False),
    ("mean_end", "Final mean opinion", False),
    ("model_over_identity", "Prediction MAE / identity MAE", True),
    ("effective_neighbors", "Mean effective neighbors per row", True),
]:
    fig, ax = plt.subplots(figsize=(7.0, 4.5))
    base_y = float(ref[metric].mean())
    ax.axhline(base_y, linestyle="--", linewidth=1.0, label="none")

    for kind in family_order:
        g = (
            work[work["regularizer_kind"] == kind]
            .groupby("regularizer_lambda", as_index=False)[metric]
            .mean()
            .sort_values("regularizer_lambda")
        )
        ax.plot(g["regularizer_lambda"], g[metric], marker="o", label=kind)

    ax.set_xscale("log")
    ax.set_xlabel("Regularization strength λ")
    ax.set_ylabel(ylabel)
    ax.set_title(
        f"{ylabel} vs regularization strength"
        + (" (lower is better)" if lower_better else " (higher is better)")
    )
    ax.legend()
    fig.tight_layout()
    plt.show()
    plt.close(fig)


## Recovery–control trade-off

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 5.0))
markers = {"none": "o", "theta_l2": "s", "entropy": "^", "gini": "D"}

for kind, g in summary.groupby("regularizer_kind"):
    ax.scatter(
        g["row_TV"], g["mean_end"],
        marker=markers[kind], s=65, label=kind,
    )
    for _, r in g.iterrows():
        label = "0" if kind == "none" else f"{r['regularizer_lambda']:.0e}"
        ax.annotate(
            label,
            (r["row_TV"], r["mean_end"]),
            xytext=(4, 3), textcoords="offset points", fontsize=8,
        )

ax.set_xlabel("Mean row-TV error of A (lower is better)")
ax.set_ylabel("Final mean opinion (higher is better)")
ax.set_title("Graph recovery vs downstream control")
ax.legend()
fig.tight_layout()
plt.show()
plt.close(fig)


## Focused Gini refinement

This table isolates the final Gini grid. `lambda_label` is kept in scientific
notation so the small values are not rounded to zero.

The purpose is to locate the recovery/control trade-off, not to optimize a
hidden scalar score.


In [ ]:
gini_summary = (
    summary[summary["regularizer_kind"] == "gini"]
    .copy()
    .sort_values("regularizer_lambda")
)
gini_summary.insert(
    2,
    "lambda_label",
    gini_summary["regularizer_lambda"].map(lambda x: f"{x:.1e}")
)

display(
    gini_summary[
        [
            "lambda_label",
            "mean_end", "delta_control",
            "prediction_ratio",
            "row_TV", "delta_row_TV",
            "true_edge_mass", "edge_AP",
            "centrality_L1", "effective_neighbors",
        ]
    ].round(6)
)

print("\nGini — structured HK only:")
gini_hk = hk[hk["regularizer_kind"] == "gini"].copy().sort_values("regularizer_lambda")
gini_hk.insert(
    2,
    "lambda_label",
    gini_hk["regularizer_lambda"].map(lambda x: f"{x:.1e}")
)
display(
    gini_hk[
        [
            "lambda_label", "control", "delta_control",
            "prediction_ratio", "row_TV", "delta_row_TV",
            "true_edge_mass", "edge_AP", "centrality_L1",
            "effective_neighbors",
        ]
    ].round(6)
)

# Gini-only Pareto frontier for row-TV vs control.
gp = gini_summary.copy()
gini_pareto_idx = []
for i, r in gp.iterrows():
    dominated = False
    for j, q in gp.iterrows():
        if i == j:
            continue
        if (
            q["row_TV"] <= r["row_TV"]
            and q["mean_end"] >= r["mean_end"]
            and (
                q["row_TV"] < r["row_TV"]
                or q["mean_end"] > r["mean_end"]
            )
        ):
            dominated = True
            break
    if not dominated:
        gini_pareto_idx.append(i)

gini_pareto = gp.loc[gini_pareto_idx].sort_values("regularizer_lambda")
print("\nGini-only Pareto candidates:")
tmp = gini_pareto[
    [
        "regularizer_lambda", "row_TV", "true_edge_mass",
        "edge_AP", "mean_end", "prediction_ratio", "effective_neighbors",
    ]
].copy()
tmp.insert(1, "lambda_label", tmp["regularizer_lambda"].map(lambda x: f"{x:.1e}"))
display(tmp.drop(columns=["regularizer_lambda"]).round(6))


## Edge-of-grid check

If the strongest observed setting for a family occurs at the smallest or
largest tested \(\lambda\), that family may need a small refinement before the
three-seed confirmation.


In [ ]:
grid_rows = []

for kind in ["theta_l2", "entropy", "gini"]:
    g = summary[summary["regularizer_kind"] == kind].sort_values("regularizer_lambda")
    lambdas = g["regularizer_lambda"].to_numpy(float)

    for criterion, series, direction in [
        ("row_TV", g["row_TV"].to_numpy(float), "min"),
        ("control", g["mean_end"].to_numpy(float), "max"),
        ("true_edge_mass", g["true_edge_mass"].to_numpy(float), "max"),
    ]:
        idx = int(np.argmin(series) if direction == "min" else np.argmax(series))
        grid_rows.append({
            "regularizer_kind": kind,
            "criterion": criterion,
            "best_lambda": lambdas[idx],
            "best_value": series[idx],
            "at_lower_grid_edge": idx == 0,
            "at_upper_grid_edge": idx == len(lambdas) - 1,
        })

edge_check = pd.DataFrame(grid_rows)
display(edge_check.round(8))


## Compact hand-off table

In [ ]:
compact = summary[
    [
        "regularizer_kind", "regularizer_lambda",
        "mean_end", "delta_control",
        "prediction_ratio",
        "row_TV", "delta_row_TV",
        "true_edge_mass", "edge_AP",
        "centrality_L1", "effective_neighbors",
    ]
].sort_values(["regularizer_kind", "regularizer_lambda"])

display(compact.round(6))

print("=" * 92)
print("REGULARIZATION SWEEP ANALYSIS COMPLETE")
print("=" * 92)
print("Conditions:", len(summary))
print("Paired environments per condition: 195")
print("Learning seed: 0 (coarse tuning)")
print()
print("Next decision after inspection:")
print("  - refine lambda if optimum is on a grid edge, OR")
print("  - validate the strongest candidate(s) with learning seeds 0,1,2.")
